<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/PDB_Subunit_Search_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Colab setup
!pip -q install biopython pandas

from pathlib import Path
import re
import pandas as pd

from Bio import SeqIO
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices
from Bio.SeqRecord import SeqRecord

**Cell 2 — Mount Drive and define folders**

In [6]:
from pathlib import Path
from google.colab import drive

# ----------------------------
# 1) Mount Google Drive safely
# ----------------------------
DRIVE_MOUNT = Path("/content/drive")
MYDRIVE = DRIVE_MOUNT / "MyDrive"

if not MYDRIVE.exists():
    drive.mount(str(DRIVE_MOUNT))
else:
    print("Google Drive already mounted.")

# ----------------------------
# 2) User-configurable settings
# ----------------------------
SCRATCH_ROOT = MYDRIVE / "_Scratch"
PROJECT_NAME = "PDB_Subunit_Search"

# Set the target subunit here
TARGET_SUBUNIT = "NdhA"

# Standard query filename pattern
QUERY_FILENAME = f"{TARGET_SUBUNIT}_query.fa"

# Allowed FASTA-like extensions
FASTA_EXTENSIONS = {".fa", ".faa", ".fasta", ".fas", ".txt"}

# ----------------------------
# 3) Project directory layout
# ----------------------------
PROJECT_DIR = SCRATCH_ROOT / PROJECT_NAME

DATA_DIR = PROJECT_DIR / "Data"
QUERY_DIR = DATA_DIR / "Query"
PDB_FASTA_DIR = DATA_DIR / "Structure_FASTAs"

OUTPUT_DIR = PROJECT_DIR / "Outputs"
LOG_DIR = PROJECT_DIR / "Logs"

# Create all expected folders
for d in [SCRATCH_ROOT, PROJECT_DIR, DATA_DIR, QUERY_DIR, PDB_FASTA_DIR, OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 4) Key file paths
# ----------------------------
QUERY_FASTA = QUERY_DIR / QUERY_FILENAME

SUMMARY_TSV = OUTPUT_DIR / f"{TARGET_SUBUNIT}_best_hits_summary.tsv"
BEST_HITS_FASTA = OUTPUT_DIR / f"{TARGET_SUBUNIT}_best_hits_per_structure.fa"

# Optional future outputs
HIT_LOG = LOG_DIR / f"{TARGET_SUBUNIT}_run_log.txt"

# ----------------------------
# 5) Report current state
# ----------------------------
print("[Project configuration]")
print("SCRATCH_ROOT  :", SCRATCH_ROOT)
print("PROJECT_DIR   :", PROJECT_DIR)
print("TARGET_SUBUNIT:", TARGET_SUBUNIT)
print()

print("[Input locations]")
print("QUERY_DIR     :", QUERY_DIR)
print("QUERY_FASTA   :", QUERY_FASTA)
print("PDB_FASTA_DIR :", PDB_FASTA_DIR)
print()

print("[Output locations]")
print("OUTPUT_DIR    :", OUTPUT_DIR)
print("SUMMARY_TSV   :", SUMMARY_TSV)
print("BEST_HITS_FASTA:", BEST_HITS_FASTA)
print("LOG_DIR       :", LOG_DIR)
print()

# ----------------------------
# 6) Basic validation / status
# ----------------------------
if QUERY_FASTA.exists():
    print(f"Query file found: {QUERY_FASTA.name}")
else:
    print(f"Query file NOT found: {QUERY_FASTA.name}")
    print(f"Please place your query FASTA in: {QUERY_DIR}")

pdb_fasta_files = sorted(
    [p for p in PDB_FASTA_DIR.iterdir() if p.is_file() and p.suffix.lower() in FASTA_EXTENSIONS]
)

print(f"Structure FASTA files found: {len(pdb_fasta_files)}")
for p in pdb_fasta_files[:10]:
    print("  -", p.name)

if len(pdb_fasta_files) == 0:
    print(f"No FASTA-like files found yet in: {PDB_FASTA_DIR}")

Google Drive already mounted.
[Project configuration]
SCRATCH_ROOT  : /content/drive/MyDrive/_Scratch
PROJECT_DIR   : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search
TARGET_SUBUNIT: NdhA

[Input locations]
QUERY_DIR     : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Data/Query
QUERY_FASTA   : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Data/Query/NdhA_query.fa
PDB_FASTA_DIR : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Data/Structure_FASTAs

[Output locations]
OUTPUT_DIR    : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs
SUMMARY_TSV   : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhA_best_hits_summary.tsv
BEST_HITS_FASTA: /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhA_best_hits_per_structure.fa
LOG_DIR       : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Logs

Query file NOT found: NdhA_query.fa
Please place your query FASTA in: /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Data/Query
Structure FASTA files 

**Cell 3 — Alignment/scoring setup**

In [3]:
# Protein alignment setup
aligner = PairwiseAligner()
aligner.mode = "local"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

def clean_protein_sequence(seq: str) -> str:
    """
    Keep only likely amino-acid letters and remove whitespace/gap symbols.
    """
    seq = str(seq).upper()
    seq = re.sub(r"[^A-Z]", "", seq)
    return seq

def percent_identity_from_alignment(alignment, seq1: str, seq2: str) -> float:
    """
    Estimate percent identity over aligned positions from a PairwiseAligner alignment.
    """
    aligned1 = alignment.aligned[0]
    aligned2 = alignment.aligned[1]

    matches = 0
    aligned_len = 0

    for (s1, e1), (s2, e2) in zip(aligned1, aligned2):
        frag1 = seq1[s1:e1]
        frag2 = seq2[s2:e2]
        n = min(len(frag1), len(frag2))
        aligned_len += n
        matches += sum(a == b for a, b in zip(frag1[:n], frag2[:n]))

    if aligned_len == 0:
        return 0.0
    return 100.0 * matches / aligned_len

**Cell 3 — Helper functions for naming and sequence cleanup**

In [7]:
import re
from pathlib import Path
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def clean_protein_sequence(seq: str) -> str:
    seq = str(seq).upper()
    seq = re.sub(r"[^A-Z]", "", seq)
    return seq

def query_label_from_filename(path: Path) -> str:
    """
    Example:
      NdhC_Telong.fas -> NDHC
      NdhA_query.fa   -> NDHA
    """
    stem = path.stem

    # remove common suffixes
    stem = re.sub(r"(?i)_telong$", "", stem)
    stem = re.sub(r"(?i)_query$", "", stem)
    stem = re.sub(r"(?i)_protein$", "", stem)

    return stem.upper()

def structure_label_from_filename(path: Path) -> str:
    """
    Example:
      4HEA-Thermus.fasta -> 4HEA-thermus
      7R41-BOVINE.fasta  -> 7R41-bovine
    """
    stem = path.stem.strip()
    return stem.lower()

def output_fasta_name(query_path: Path) -> str:
    """
    Example:
      NdhC_Telong.fas -> NDHC_structure_hits.fa
    """
    q = query_label_from_filename(query_path)
    return f"{q}_structure_hits.fa"

def output_summary_name(query_path: Path) -> str:
    """
    Example:
      NdhC_Telong.fas -> NDHC_structure_hits_summary.tsv
    """
    q = query_label_from_filename(query_path)
    return f"{q}_structure_hits_summary.tsv"

**Cell 4 — Alignment setup**

In [8]:
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices

aligner = PairwiseAligner()
aligner.mode = "local"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

def percent_identity_from_alignment(alignment, seq1: str, seq2: str) -> float:
    aligned1 = alignment.aligned[0]
    aligned2 = alignment.aligned[1]

    matches = 0
    aligned_len = 0

    for (s1, e1), (s2, e2) in zip(aligned1, aligned2):
        frag1 = seq1[s1:e1]
        frag2 = seq2[s2:e2]
        n = min(len(frag1), len(frag2))
        aligned_len += n
        matches += sum(a == b for a, b in zip(frag1[:n], frag2[:n]))

    if aligned_len == 0:
        return 0.0
    return 100.0 * matches / aligned_len

**Cell 5 — Best hit finder for one structure FASTA**

In [9]:
def best_hit_in_structure_fasta(query_seq: str, query_label: str, fasta_path: Path) -> dict | None:
    """
    Find the best homolog in one structure FASTA file.
    Returns a dict with metadata and renamed output ID.
    """
    best = None
    structure_label = structure_label_from_filename(fasta_path)

    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        target_seq = clean_protein_sequence(rec.seq)
        if not target_seq:
            continue

        score = aligner.score(query_seq, target_seq)
        if score <= 0:
            continue

        aln = aligner.align(query_seq, target_seq)[0]
        pid = percent_identity_from_alignment(aln, query_seq, target_seq)

        renamed_id = f"{query_label}_{structure_label}"

        row = {
            "query_label": query_label,
            "structure_file": fasta_path.name,
            "structure_label": structure_label,
            "renamed_id": renamed_id,
            "hit_id": rec.id,
            "hit_description": rec.description,
            "hit_length": len(target_seq),
            "score": score,
            "percent_identity": pid,
            "sequence": target_seq,
        }

        if best is None or row["score"] > best["score"]:
            best = row

    return best

**Cell 6 — Run all queries against all structure FASTAs**

In [12]:
from pathlib import Path
from Bio import SeqIO

def query_label_from_filename(path: Path) -> str:
    """
    Example:
      NdhA_Telong.fas -> NdhA
      NdhC_query.fa   -> NdhC
      ndhh.fa         -> NdhH
    """
    stem = path.stem

    # remove common suffixes
    stem = stem.replace("_Telong", "")
    stem = stem.replace("_query", "")
    stem = stem.replace("_protein", "")

    lower = stem.lower()
    if lower.startswith("ndh") and len(stem) >= 4:
        return "Ndh" + stem[3:].upper()

    return stem

def structure_label_from_filename(path: Path) -> str:
    """
    Example:
      4HEA-Thermus.fasta      -> 4HEA-Thermus
      7R41-BOVINE.fasta       -> 7R41-Bovine
      8B9Z-DROSOPHILA.fasta   -> 8B9Z-Drosophila
      6HUM_Thermovestidis.fa  -> 6HUM-Thermovestidis
    """
    stem = path.stem.strip()
    stem = stem.replace("_", "-")

    if "-" in stem:
        pdb, org = stem.split("-", 1)
        return f"{pdb.upper()}-{org.capitalize()}"
    return stem

def best_hit_in_structure_fasta(query_seq: str, query_label: str, fasta_path: Path) -> dict | None:
    """
    Find the best homolog in one structure FASTA file.
    Returns a dict with metadata and simplified renamed output ID.
    """
    best = None
    structure_label = structure_label_from_filename(fasta_path)

    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        target_seq = clean_protein_sequence(rec.seq)
        if not target_seq:
            continue

        score = aligner.score(query_seq, target_seq)
        if score <= 0:
            continue

        aln = aligner.align(query_seq, target_seq)[0]
        pid = percent_identity_from_alignment(aln, query_seq, target_seq)

        renamed_id = f"{query_label}-{structure_label}"

        row = {
            "query_label": query_label,
            "structure_file": fasta_path.name,
            "structure_label": structure_label,
            "renamed_id": renamed_id,
            "hit_id": rec.id,
            "hit_description": rec.description,
            "hit_length": len(target_seq),
            "score": score,
            "percent_identity": pid,
            "sequence": target_seq,
        }

        if best is None or row["score"] > best["score"]:
            best = row

    return best

# -----------------------------
# Run all queries against all structure FASTA files
# -----------------------------
query_files = sorted(
    [p for p in QUERY_DIR.iterdir() if p.is_file() and p.suffix.lower() in FASTA_EXTENSIONS]
)

structure_files = sorted(
    [p for p in PDB_FASTA_DIR.iterdir() if p.is_file() and p.suffix.lower() in FASTA_EXTENSIONS]
)

print(f"Found {len(query_files)} query files")
print(f"Found {len(structure_files)} structure FASTA files")

assert len(query_files) > 0, f"No query FASTA files found in {QUERY_DIR}"
assert len(structure_files) > 0, f"No structure FASTA files found in {PDB_FASTA_DIR}"

all_results = {}

for query_path in query_files:
    query_records = list(SeqIO.parse(str(query_path), "fasta"))
    assert len(query_records) >= 1, f"No sequences found in query file {query_path}"

    query_rec = query_records[0]
    query_seq = clean_protein_sequence(query_rec.seq)
    query_label = query_label_from_filename(query_path)

    print(f"\n[Query] {query_path.name} -> {query_label}")

    results = []
    for fasta_file in structure_files:
        hit = best_hit_in_structure_fasta(query_seq, query_label, fasta_file)
        if hit is None:
            print(f"  [no hit]  {fasta_file.name}")
            continue
        results.append(hit)
        print(
            f"  [best]    {fasta_file.name} -> {hit['renamed_id']}  "
            f"score={hit['score']:.1f}  pid={hit['percent_identity']:.1f}"
        )

    all_results[query_label] = results

Found 5 query files
Found 11 structure FASTA files

[Query] NdhA_Telong.fas -> NdhA
  [best]    4HEA-Thermus.fasta -> NdhA-4HEA-Thermus  score=619.0  pid=44.9
  [best]    6G2J-MOUSE.fasta -> NdhA-6G2J-Mouse  score=477.5  pid=40.0
  [best]    6HUM_Thermosynechoccus_vestidis.fasta -> NdhA-6HUM-Thermosynechoccus-vestidis  score=1873.0  pid=100.0
  [best]    6RFR-YARROWIA.fasta -> NdhA-6RFR-Yarrowia  score=609.5  pid=40.9
  [best]    7A23-PLANT-Mito.fasta -> NdhA-7A23-Plant-mito  score=623.5  pid=43.8
  [best]    7EU3_Barley_Chloroplast.fasta -> NdhA-7EU3-Barley-chloroplast  score=1126.0  pid=62.7
  [best]    7R41-BOVINE.fasta -> NdhA-7R41-Bovine  score=501.5  pid=38.7
  [best]    7Z80-E_COLI.fasta -> NdhA-7Z80-E-coli  score=631.0  pid=42.4
  [best]    7ZD6-OVIS.fasta -> NdhA-7ZD6-Ovis  score=492.5  pid=41.5
  [best]    8B9Z-DROSOPHILA.fasta -> NdhA-8B9Z-Drosophila  score=414.0  pid=37.5
  [best]    8QBY-PARACOCCUS.fasta -> NdhA-8QBY-Paracoccus  score=609.5  pid=40.9

[Query] NdhC_Telong.f

**Cell 7 — Write one FASTA and one summary per query**

In [13]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd

for query_label, results in all_results.items():
    if len(results) == 0:
        print(f"[skip] No hits found for {query_label}")
        continue

    # Summary table keeps the full provenance/details
    df = pd.DataFrame(results).drop(columns=["sequence"])
    df = df.sort_values(["structure_file"]).reset_index(drop=True)

    summary_path = OUTPUT_DIR / f"{query_label}_structure_hits_summary.tsv"
    df.to_csv(summary_path, sep="\t", index=False)

    # Output FASTA: simplified headers only
    out_records = []
    for r in results:
        out_records.append(
            SeqRecord(
                Seq(r["sequence"]),
                id=r["renamed_id"],
                description=""   # critical: prevents verbose FASTA headers
            )
        )

    fasta_path = OUTPUT_DIR / f"{query_label}_structure_hits.fa"
    SeqIO.write(out_records, str(fasta_path), "fasta")

    print(f"\n[Wrote] {query_label}")
    print("  FASTA  :", fasta_path)
    print("  SUMMARY:", summary_path)


[Wrote] NdhA
  FASTA  : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhA_structure_hits.fa
  SUMMARY: /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhA_structure_hits_summary.tsv

[Wrote] NdhC
  FASTA  : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhC_structure_hits.fa
  SUMMARY: /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhC_structure_hits_summary.tsv

[Wrote] NdhE
  FASTA  : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhE_structure_hits.fa
  SUMMARY: /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhE_structure_hits_summary.tsv

[Wrote] NdhG
  FASTA  : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhG_structure_hits.fa
  SUMMARY: /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhG_structure_hits_summary.tsv

[Wrote] NdhH
  FASTA  : /content/drive/MyDrive/_Scratch/PDB_Subunit_Search/Outputs/NdhH_structure_hits.fa
  SUMMARY: /content/drive/MyDrive/_Scratch/PDB_Subunit_Se